# DeepFace ライブラリによる顔の類似度ヒートマップの作成例

This Person Does Not Exist から取得した人物の顔写真から、類似度を計算してヒートマップ化

## 注意

ローカルで ipynb を実行するために ipykernel のインストールが必要です。
```
$ pip install ipykernel
```
を実行してからノートを実行してください。

In [ ]:
# パッケージをインストール
!pip install deepface tf_keras matplotlib seaborn pillow

In [ ]:
# 人物画像をダウンロード
import requests

url = "https://thispersondoesnotexist.com/"
num_images = 6

for i in range(1, num_images + 1):
    file_name = f"person_{i}.jpg"
    try:
        print(f"{i}枚目: '{url}' から画像をダウンロードしています...")
        response = requests.get(url)
        response.raise_for_status()
        with open(file_name, 'wb') as f:
            f.write(response.content)
        print(f"画像 '{file_name}' のダウンロードが完了しました。")
    except requests.exceptions.RequestException as e:
        print(f"{i}枚目の画像のダウンロードに失敗しました: {e}")


In [ ]:
# 顔画像の類似度を計算

import glob

# カレントディレクトリのjpgファイル一覧を取得
images = sorted(glob.glob("*.jpg"))

from deepface import DeepFace
import numpy as np


# 類似度行列を計算
num = len(images)
similarity_matrix = np.zeros((num, num))


for i in range(num):
    for j in range(num):
        if i == j:
            similarity_matrix[i, j] = 1.0
        elif i < j:
            # 顔画像間の類似度を計算
            print(f"Calculating similarity between {images[i]} and {images[j]}...")
            result = DeepFace.verify(img1_path=images[i], img2_path=images[j],model_name='Facenet', enforce_detection=True)
            similarity = abs(1 - result['distance'])  # 距離が小さいほど類似
            similarity_matrix[i, j] = similarity
            similarity_matrix[j, i] = similarity


In [ ]:
# 類似度行列をヒートマップとして可視化

from PIL import Image
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import matplotlib.pyplot as plt


thumbnails = []
for img_path in images:
    img = Image.open(img_path)
    img.thumbnail((32, 32))
    thumbnails.append(img)
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(similarity_matrix, cmap='viridis', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Similarity')

# サムネイルをx軸とy軸のticksに表示
for idx, thumbnail in enumerate(thumbnails):
    imagebox = OffsetImage(thumbnail, zoom=1)
    ab = AnnotationBbox(imagebox, (idx, -0.5), frameon=False, box_alignment=(0.5, 0))
    ax.add_artist(ab)
    ab = AnnotationBbox(imagebox, (-0.5, idx), frameon=False, box_alignment=(1, 0.5))
    ax.add_artist(ab)

ax.set_xticks(range(num))
ax.set_yticks(range(num))
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.set_title("Similarity Matrix of Faces", pad=45)

# 類似度値を各セルに表示
for i in range(num):
    for j in range(num):
        ax.text(j, i, f"{similarity_matrix[i, j]:.2f}", ha="center", va="center", color="black")

plt.tight_layout()
plt.show()